## **A Visual Analysis of Artificial Intelligence in Medicine**
**Trends, Paradigms, and Collaboration Networks**

**Student:** Diriczi Zsolt-Csaba  
**Course:** Big Data Visualization  

### **Notebook Structure**

This notebook is organized into six main sections:

0. **Data Collection and Preprocessing**: builds the dataset from OpenAlex metadata and applies additional filtering to remove false-positive publications.
1. **Publication Growth**: analyzes how the number of AI in healthcare publications increased over time.
2. **Paradigm and Method Shift Over Time**: analyzes how AI method terminology changed across historical periods using predefined method keywords and word clouds.
3. **Geographic Distribution of Publications**: visualizes where AI in healthcare research is produced using a country-level map and the yearly publication share of the top countries.
4. **Medical–Technical Institution Collaboration**: measures collaboration between medical and non-clinical institutions, both overall and over time.
5. **Country Collaboration Network**: builds country level nodes and edges for Gephi, identifies the strongest collaboration pairs, and analyzes international collaboration over time.

### **0. Dataset Construction and Filtering**

#### **0.1 Dataset Construction**

The first step of the project is to construct the dataset using the `extract_dataset.py` script. This script collects publication metadata from the OpenAlex API using a query that searches for papers containing both medical terms and AI related terms. Also in this extraction script any missing values are filled with empty string or default value where numbers

In [1]:
!python3 utils/extract_dataset.py

/Users/diriczizsolt-csaba/Desktop/UPT/BDV/Project/.env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Starting from page 1 with cursor '*'. Already saved papers: 0
Fetched page 1, received 100 papers, saved 100 new rows, total saved: 100
Fetched page 2, received 100 papers, saved 100 new rows, total saved: 200
Fetched page 3, received 100 papers, saved 100 new rows, total saved: 300
Fetched page 4, received 100 papers, saved 100 new rows, total saved: 400
Fetched page 5, received 100 papers, saved 100 new rows, total saved: 500
^C
Traceback (most recent call last):
  File "/Users/diriczizsolt-csaba/Desktop/UPT/BDV/Project/utils/extract_dataset.py", line 360, in <module>
    fetch_publications(
  File "/Users/diriczizsolt-csaba/Desktop/UPT/BDV/Project/utils/extract_dataset.py", line 350, in fet

#### **0.2 Dataset Filtering**

However, the OpenAlex request query cannot fully enforce the complete keyword combination filtering logic needed for this project. Because of this, the initial dataset is first extracted from OpenAlex, and then an additional filtering step is applied locally to remove false-positive publications.

Two main types of false positives were identified during a closer dataset inspection. The first situation involved papers published before 2017 that contained the word *transformer*. In many of these publications, the term referred to electrical transformers rather than the Transformer architecture used in modern artificial intelligence. The second case involved papers about plant disease detection. These papers matched the healthcare related keyword *disease*, but they referred to agricultural or plant related applications rather than human healthcare.

Therefore, after the initial dataset was collected, an additional filtering layer was applied to remove these false positives before generating the visualizations.

In [2]:
import re
import pandas as pd

In [3]:
TEXT_COLUMNS = ["title", "abstract", "keywords"]

In [4]:
DATASET_PATH = "data/openalex_ai_healthcare_publications_1980_2026.csv"

In [5]:
publications_df = pd.read_csv(DATASET_PATH)

In [ ]:
for col in TEXT_COLUMNS:
    if col not in publications_df.columns:
        publications_df[col] = ""

publications_df["combined_text"] = (
    publications_df[TEXT_COLUMNS]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

transformer_pattern = r"\btransformer(s)?\b"

pre_2017_transformer_mask = (
    (publications_df["publication_year"] < 2017)
    & publications_df["combined_text"].str.contains(
        transformer_pattern,
        case=False,
        regex=True,
        na=False
    )
)

plant_disease_pattern = (
    r"\bplant disease(s)?\b|"
    r"\bplant pathogen(s)?\b|"
    r"\bcrop disease(s)?\b|"
    r"\bleaf disease(s)?\b|"
    r"\bplant leaf\b|"
    r"\bcrop leaf\b|"
    r"\bagricultural disease(s)?\b"
)

plant_disease_mask = publications_df["combined_text"].str.contains(plant_disease_pattern, case=False, regex=True, na=False)

In [11]:
remove_mask = pre_2017_transformer_mask | plant_disease_mask

print("Pre-2017 transformer papers removed:", pre_2017_transformer_mask.sum())
print("Plant-related disease papers removed:", plant_disease_mask.sum())
print("Total papers removed:", remove_mask.sum())

publications_df = publications_df.loc[~remove_mask].copy()

publications_df = publications_df.drop(columns=["combined_text"], errors="ignore")

print("Remaining papers:", len(publications_df))

Pre-2017 transformer papers removed: 4994
Plant-related disease papers removed: 13039
Total papers removed: 18032
Remaining papers: 853899


In [13]:
publications_df.to_csv(
    "data/openalex_ai_healthcare_publications_1980_2026_filtered_2.csv",
    index=False,
    encoding="utf-8"
)